In [ ]:
# Toda célula de um caderno marimo é uma função Python; o que ela devolve no "return" fica para as outras células.
import marimo as mo

In [ ]:
# O PAINEL: a história em português simples e, numa caixa fechada, os detalhes técnicos (lê o PIPELINE.md).
import sys as _sys
_pasta = str(mo.notebook_dir())
if _pasta not in _sys.path:
    _sys.path.insert(0, _pasta)
import painel
painel.mostrar(mo)

# Ciclo 2, etapa 13: onde o 6B erra

**Em português:** no simulado, o 6B acerta 85 de cada 100 fontes que mostra e acha 8 de cada 10 fontes do mesmo fato.
Aqui queremos saber **quem são** os outros 15 e os 2 que ficam de fora: se os erros se espalham ao acaso ou se
concentram num tipo de fonte. Erro de extração se espalha; fenômeno se concentra. Se se concentram, é ali que o app
deve ter cuidado, ou que o próximo ciclo deve mirar. Tudo **no simulado**: a prova continua no cofre e só entra no
caderno na etapa 14.

| O que olhamos | Em português | Em termos técnicos |
|---|---|---|
| **É o mesmo modelo?** | treinamos de novo o 6B a partir do registro da etapa 12 e conferimos que dá as mesmas notas | reprodutibilidade, impressão digital das notas |
| **O 6B e a regra, caso a caso** | das fontes do mesmo fato: quantas os dois acham, quantas só um acha; dos enganos, quantos são dos dois | tabela cruzada das decisões |
| **Os piores erros, um a um** | as mostradas com nota alta que não eram o mesmo fato; e as que eram e ficaram de fora com nota baixa | falsos positivos e falsos negativos |
| **Erros por recorte** | erra mais com fonte velha? com algum buscador? com algum tipo de manchete? quando a regra não comparou? | métricas por recorte, todos os que a tabela permite |

**O que eu espero, escrito antes de rodar:**

| # | Expectativa | O que faria eu mudar de ideia |
|---|---|---|
| 1 | os mostrados por engano são, na maioria (mais de metade dos 15 piores), o **mesmo tipo de acontecimento em outra data ou lugar** (outra chegada de remédio, outro corpo identificado) | se a maioria for assunto diferente: o 6B confunde tema, não fato |
| 2 | os deixados de fora se concentram em fonte com **mais de um mês** e em "só relevância" | se se espalharem por idade e tipo de busca |
| 3 | nos rótulos duvidosos ("maioria de três"), o 6B erra pelo menos o dobro | se errar igual: os duvidosos não são mais difíceis |
| 4 | dos enganos do 6B, a regra também aponta no máximo 1 em cada 3 (a regra é cuidadosa) | se a regra apontar a maioria: os enganos são do rótulo ou do problema, não do 6B |
| 5 | das fontes do mesmo fato que a regra acha, o 6B acha pelo menos 95% | se a regra achar muitas que o 6B perde: o 6B não "contém" a regra |
| 6 | o Bing na web (`bing_web`) tem a pior precisão entre os buscadores | se for outro |

In [ ]:
import json
import os
import sys
import tempfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Os módulos do pipeline moram na pasta deste caderno.
if str(mo.notebook_dir()) not in sys.path:
    sys.path.insert(0, str(mo.notebook_dir()))
import guard
from dados import carregar
from medidas import precisao_e_cobertura
from preprocessamento import montar_preprocessador

estudo, simulado = carregar()          # a prova não sai daqui
escolhido = json.loads((mo.notebook_dir() / "modelo_escolhido.json").read_text(encoding="utf-8"))
MESMO_DA_REGRA = ["mesmo_acontecimento", "detalhe_divergente"]

# A CHAVE DE TESTE (o Matheus roda; a sessão de IA só confere que o código não quebra): com ML_AMOSTRA_TESTE ligada,
# poucas notícias inteiras e saída descartável; nenhum número dessa rodada vale.
AMOSTRA_TESTE = int(os.environ.get("ML_AMOSTRA_TESTE", "0"))
PASTA_SAIDA = mo.notebook_dir()
if AMOSTRA_TESTE:
    PASTA_SAIDA = Path(tempfile.mkdtemp(prefix="teste_caderno_"))

    def _poucas(tabela, alvo):
        # notícias inteiras, em ordem fixa, até juntar uns `alvo` exemplos
        _guardar, _soma = [], 0
        for _g, _n in tabela.groupby("grupo").size().sort_index().items():
            _guardar.append(_g)
            _soma += _n
            if _soma >= alvo:
                break
        return tabela[tabela["grupo"].isin(_guardar)]
    estudo = _poucas(estudo, AMOSTRA_TESTE)
    simulado = _poucas(simulado, AMOSTRA_TESTE)

In [ ]:
import hashlib
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# A NOTA DO TEXTO, como na etapa 10: no estudo, a de um modelo de linguagem que não viu o par (fora da dobra); no
# simulado, a do modelo treinado com o estudo inteiro. reindex: a ordem dos pares de cada monte.
_nota_est = pd.read_csv(mo.notebook_dir() / "notas_cross_encoder_estudo_dobras.csv",
                        index_col="id")["nota"].reindex(estudo.index).to_numpy()
_nota_sim = pd.read_csv(mo.notebook_dir() / "notas_cross_encoder_simulado.csv",
                        index_col="id")["nota"].reindex(simulado.index).to_numpy()

# O 6B CONGELADO, treinado de novo a partir do registro da etapa 12 (mesma configuração, mesmo estudo).
_peso = np.where((estudo["origem"] == "maioria de três").to_numpy(), escolhido["peso_dos_duvidosos"], 1.0)
modelo_escolhido = Pipeline([
    ("preproc", montar_preprocessador(com_regra=False, extras=("nota_texto",))),
    ("logistica", LogisticRegression(C=escolhido["C"], class_weight="balanced", max_iter=5000)),
])
modelo_escolhido.fit(estudo.assign(nota_texto=_nota_est)[escolhido["entradas"]], estudo["mesmo"],
                     logistica__sample_weight=_peso)
nota = modelo_escolhido.predict_proba(simulado.assign(nota_texto=_nota_sim)[escolhido["entradas"]])[:, 1]
aponta = nota >= escolhido["corte"]        # o corte congelado na etapa 12, sem escolher de novo

# A CONFERÊNCIA: as notas no simulado têm de ter a mesma impressão digital gravada na etapa 12.
_digital = hashlib.sha256(np.round(nota, 6).tobytes()).hexdigest()
mesmo_modelo = _digital == escolhido["impressao_digital_das_notas_no_simulado"]
if not AMOSTRA_TESTE:                      # no teste pequeno as notas são outras, de propósito
    assert mesmo_modelo, "o 6B treinado de novo NÃO dá as mesmas notas da etapa 12: não é o modelo aprovado"
nota_texto_simulado = _nota_sim
"mesmas notas da etapa 12: é o modelo aprovado" if mesmo_modelo else "(teste pequeno: conferência pulada)"

In [ ]:
# A TABELA DOS ERROS: cada fonte do simulado com o que o 6B fez, o que a regra fez e o que era de verdade.
_y = simulado["mesmo"].to_numpy()
_regra = simulado["regra"].isin(MESMO_DA_REGRA).to_numpy()
erros = simulado.assign(
    nota=np.round(nota, 3),
    nota_do_texto=np.round(nota_texto_simulado, 3),
    modelo_aponta=aponta,
    regra_aponta=_regra,
    # O tipo de cada caso, em português: o que o leitor veria.
    caso=np.select([aponta & _y, aponta & ~_y, ~aponta & _y],
                   ["acerto mostrado", "mostrado por engano", "deixado de fora"], default="acerto silencioso"),
    # A idade da fonte em faixas (dias entre a notícia e a fonte): mesmo dia ou véspera, até um mês, mais, sem data.
    # cat.add_categories + fillna: as fontes sem data viram a faixa "sem data" (no pandas 3, astype(str) deixaria
    # o vazio vazio, e o agrupamento por faixa as descartaria calado; foi o que aconteceu na primeira rodada).
    faixa_de_idade=pd.cut(simulado["idade_dias"], [-10, 1, 30, 10**6],
                          labels=["mesmo dia ou véspera", "2 a 30 dias", "mais de um mês"])
                   .cat.add_categories("sem data").fillna("sem data").astype(str),
    tipo_de_busca=np.where(simulado["comparou"] == 1, "a regra comparou", "só relevância"),
)
erros

In [ ]:
# O 6B E A REGRA, CASO A CASO. Em cima, as fontes que SÃO o mesmo fato: quem acha cada uma. Embaixo, as que NÃO são:
# quem aponta por engano. pd.crosstab conta as combinações de duas colunas.
_rotulo = {True: "aponta", False: "não aponta"}
_sao = erros[erros["mesmo"] == 1]
_nao = erros[erros["mesmo"] == 0]
cruzada_mesmo = pd.crosstab(_sao["modelo_aponta"].map(_rotulo).rename("6B"),
                            _sao["regra_aponta"].map(_rotulo).rename("regra"))
cruzada_engano = pd.crosstab(_nao["modelo_aponta"].map(_rotulo).rename("6B"),
                             _nao["regra_aponta"].map(_rotulo).rename("regra"))
# As duas contas das expectativas 4 e 5, prontas.
_regra_acha = _sao[_sao["regra_aponta"]]
_enganos_6b = _nao[_nao["modelo_aponta"]]
contas_cruzadas = {
    "das fontes do mesmo fato que a regra acha, o 6B também acha": round(float(_regra_acha["modelo_aponta"].mean()), 3) if len(_regra_acha) else None,
    "dos enganos do 6B, a regra também aponta": round(float(_enganos_6b["regra_aponta"].mean()), 3) if len(_enganos_6b) else None,
}
import marimo as _mo13
_mo13.vstack([_mo13.md("**As que SÃO o mesmo fato:** quem acha cada uma"), cruzada_mesmo,
              _mo13.md("**As que NÃO são:** quem aponta por engano"), cruzada_engano, contas_cruzadas])

In [ ]:
# OS PIORES ERROS, UM A UM. Mostrados por engano com a nota mais ALTA; deixados de fora com a nota mais BAIXA.
_colunas = ["caso", "nota", "nota_do_texto", "cosseno", "afirmacao", "titulo", "regra", "tipo_de_busca",
            "faixa_de_idade", "idade_dias", "buscador", "tipo_afirmacao", "origem"]
piores_mostrados = erros[erros["caso"] == "mostrado por engano"].sort_values(
    ["nota", "id"] if "id" in erros.columns else "nota", ascending=False, kind="stable").head(15)[_colunas]
piores_deixados = erros[erros["caso"] == "deixado de fora"].sort_values("nota", kind="stable").head(15)[_colunas]
# Para arquivo, para a sessão de IA ler do disco e o diário citar; só na rodada de verdade.
if PASTA_SAIDA.name == "ml_pipeline":
    piores_mostrados.to_csv(PASTA_SAIDA / "13_piores_mostrados_por_engano.csv", lineterminator="\n")
    piores_deixados.to_csv(PASTA_SAIDA / "13_piores_deixados_de_fora.csv", lineterminator="\n")
piores_mostrados

In [ ]:
# Os deixados de fora (eram o mesmo fato, o 6B não mostrou), da nota mais baixa para cima.
piores_deixados

In [ ]:
# ERROS POR RECORTE: em cada fatia, quantas fontes, quantas do mesmo fato, quanto o 6B e a regra acham e quanto do que
# mostram está certo. TODO recorte que a tabela permite (regra 10 do repositório), não só os bonitos.
_recortes = ["tipo_de_busca", "faixa_de_idade", "buscador", "tipo_afirmacao", "origem", "titulo_cortado",
             "quem", "acao", "assunto", "quando"]
_linhas = []
for _r in _recortes:
    for _valor, _g in erros.groupby(_r):
        _p, _c = precisao_e_cobertura(_g["mesmo"], _g["modelo_aponta"])
        _pr, _cr = precisao_e_cobertura(_g["mesmo"], _g["regra_aponta"])
        _tem_mesmo, _mostra, _mostra_r = _g["mesmo"].sum(), _g["modelo_aponta"].sum(), _g["regra_aponta"].sum()
        _linhas.append({"recorte": _r, "valor": str(_valor), "fontes": len(_g), "do mesmo fato": int(_tem_mesmo),
                        "6B mostra": int(_mostra),
                        "6B: acha de cada 10": round(10 * _c, 1) if _tem_mesmo else None,
                        "6B: precisão": round(_p, 3) if _mostra else None,
                        "regra: acha de cada 10": round(10 * _cr, 1) if _tem_mesmo else None,
                        "regra: precisão": round(_pr, 3) if _mostra_r else None,
                        "pouca amostra?": "sim" if _tem_mesmo < 10 else ""})
por_recorte = pd.DataFrame(_linhas).set_index(["recorte", "valor"])
por_recorte

In [ ]:
# FIGURA DA ETAPA 13: quanto o 6B acha, de cada 10, em cada recorte com amostra suficiente, com a regra ao lado.
_ok = por_recorte[(por_recorte["pouca amostra?"] == "") & por_recorte["6B: acha de cada 10"].notna()]
_nomes = [f"{r} = {v}" for r, v in _ok.index]
_fig, _ax = plt.subplots(figsize=(8, max(3, 0.32 * len(_ok))))
_pos = list(range(len(_ok)))
_ax.barh([p - 0.2 for p in _pos], _ok["6B: acha de cada 10"], height=0.4, label="6B")
_ax.barh([p + 0.2 for p in _pos], _ok["regra: acha de cada 10"], height=0.4, label="regra do app")
_ax.set_yticks(_pos, _nomes)
_ax.invert_yaxis()
_ax.tick_params(axis="y", labelsize=7)
_ax.set_xlabel("de cada 10 fontes do mesmo fato, quantas acha (simulado)")
_ax.set_title("Ciclo 2, etapa 13: onde o 6B acha mais e onde acha menos")
_ax.legend(fontsize=7)
_pior = _ok["6B: acha de cada 10"].idxmin() if len(_ok) else ("-", "-")
_melhor = _ok["6B: acha de cada 10"].idxmax() if len(_ok) else ("-", "-")
_explicacao = (
    f"Em cada recorte com pelo menos 10 fontes do mesmo fato: onde o 6B acha menos é {_pior[0]} = {_pior[1]} "
    f"({_ok.loc[_pior, '6B: acha de cada 10'] if len(_ok) else '-'} de cada 10); onde acha mais, {_melhor[0]} = "
    f"{_melhor[1]} ({_ok.loc[_melhor, '6B: acha de cada 10'] if len(_ok) else '-'}). Recortes com menos de 10 ficam fora "
    "da figura, mas estão na tabela, marcados como pouca amostra."
)
figura_recortes = guard.fig(13, "onde_o_6b_erra", _fig, _explicacao, state_dir=PASTA_SAIDA)

# OS RESULTADOS EM ARQUIVO, para a sessão de IA ler do disco.
_pasta = PASTA_SAIDA / "resultados"
_pasta.mkdir(exist_ok=True)
(_pasta / "13_erros.json").write_text(json.dumps({
    "etapa": 13,
    "contas_cruzadas": contas_cruzadas,
    "cruzada_mesmo": {f"6B {i} | regra {j}": int(cruzada_mesmo.loc[i, j]) for i in cruzada_mesmo.index for j in cruzada_mesmo.columns},
    "cruzada_engano": {f"6B {i} | regra {j}": int(cruzada_engano.loc[i, j]) for i in cruzada_engano.index for j in cruzada_engano.columns},
    "por_recorte": {" = ".join(map(str, k)): v for k, v in por_recorte.to_dict(orient="index").items()},
}, indent=1, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, "item") else str(o)) + "\n",
    encoding="utf-8", newline="\n")
mo.vstack([mo.image(src=str(figura_recortes), width=640), mo.md(_explicacao)])

# Ciclo 2, etapa 14: a prova (o cofre abre uma vez)

**Em português:** a prova são as 4.298 fontes das 100 manchetes da rodada 4, lacradas antes das buscas e dos rótulos.
Nenhuma escolha deste ciclo olhou para elas. Agora o 6B congelado (C = 0,03, corte na nota 0,874) responde a cada
fonte, e medimos **uma vez**. O número que sair é o número, mesmo se for pior que o do simulado: sem voltar para
ajustar.

| Garantia | Em português | Em termos técnicos |
|---|---|---|
| É a pilha lacrada? | o cofre confere a impressão digital da pilha (id e rótulo) antes de abrir | `guard.final_test`, impressão digital |
| Uma vez só | o cofre conta as aberturas e recusa a segunda | `test_touches` |
| Rodar o caderno de novo não reabre | na segunda vez, o caderno só mostra o resultado gravado da primeira | `resultados/14_prova.json` |
| É o modelo aprovado | o 6B treinado de novo deu as mesmas notas da etapa 12 (célula da etapa 13) | impressão digital das notas |
| A nota do texto é do mesmo modelo de linguagem | os pesos guardados deram as mesmas notas do simulado antes de ler a prova | `treinar_cross_encoder.py prova` |

**O contrato do ciclo 2 (etapa 3), as duas exigências:** (1) o 6B acha mais fontes do mesmo fato que a regra do app,
com a faixa da diferença toda acima de zero (sorteando notícias da prova); (2) a precisão do 6B fica no máximo 3
pontos abaixo da precisão da regra. Na mesma abertura, só para informar (fora do contrato): a régua (o modelo simples
sem o modelo de linguagem, o plano B do app) e as fatias.

**O que eu espero, escrito antes de abrir:** o 6B acha de 6,5 a 8,0 de cada 10 (no simulado, 8,1, com o corte
escolhido nele: a prova sai abaixo), com precisão de 75% a 88%; a regra acha de 3,0 a 5,0, com precisão de 78% a 92%.
A exigência 1 passa com folga (diferença de +2,5 a +4,5 de cada 10). A exigência 2 é a frágil: o ruído da precisão
com 100 notícias é de uns ±10 pontos (etapa 9), e dou 6 em 10 de passar. A régua fica de 0 a 1 de cada 10 abaixo
do 6B. E as fontes com mais de um mês continuam as piores (etapa 13).

In [ ]:
from dados import carregar_prova
from medidas import passa_no_contrato
from sklearn.linear_model import LogisticRegression as _LR14
from sklearn.pipeline import Pipeline as _P14

_arquivo_resultado = mo.notebook_dir() / "resultados" / "14_prova.json"

if AMOSTRA_TESTE:
    # TESTE PEQUENO: o cofre de verdade NUNCA é tocado. Monta-se um "cofre de mentira" numa pasta descartável, com
    # uma pilha falsa tirada do simulado, só para exercitar o código.
    test = simulado[["mesmo"]].reset_index().sort_values("id", kind="stable").reset_index(drop=True)
    linhas_da_prova = simulado
    _nota_texto_prova = pd.read_csv(mo.notebook_dir() / "notas_cross_encoder_simulado.csv", index_col="id")["nota"]
    _cofre = PASTA_SAIDA
    _arquivo_resultado = PASTA_SAIDA / "resultados" / "14_prova.json"
    (_cofre / guard.STATE_FILE).write_text(json.dumps(
        {"test_fingerprint": guard._fingerprint(test), "n_test": len(test), "test_touches": 0}), encoding="utf-8")
    _ja_aberta = False
else:
    _cofre = mo.notebook_dir()
    _estado = json.loads((_cofre / guard.STATE_FILE).read_text(encoding="utf-8"))
    _ja_aberta = _estado["test_touches"] >= 1

if _ja_aberta:
    # A PROVA JÁ FOI ABERTA: só mostra o que foi gravado na única abertura. Nada é medido de novo.
    resultado_prova = json.loads(_arquivo_resultado.read_text(encoding="utf-8"))
    aviso_prova = "A prova já foi aberta uma vez; abaixo, o resultado gravado naquela abertura."
else:
    if not AMOSTRA_TESTE:
        test, linhas_da_prova = carregar_prova()
        # A nota do texto de cada fonte da prova, dada pelo treinar_cross_encoder.py prova (sem rótulo).
        _nota_texto_prova = pd.read_csv(mo.notebook_dir() / "notas_cross_encoder_prova.csv", index_col="id")["nota"]
    _linhas_com_nota = linhas_da_prova.assign(nota_texto=_nota_texto_prova.reindex(linhas_da_prova.index))
    if _linhas_com_nota["nota_texto"].isna().any():
        raise ValueError("fonte da prova sem a nota do modelo de linguagem: rode o treinar_cross_encoder.py prova")

    def prever(features):
        """O que o 6B congelado responde para cada fonte da prova: 1 = "mesmo fato", 0 = não.
        `features` é a pilha congelada sem o rótulo: só o id; as entradas vêm da tabela pronta, pelo id."""
        _l = _linhas_com_nota.loc[features["id"]]
        _nota = modelo_escolhido.predict_proba(_l[escolhido["entradas"]])[:, 1]
        return (_nota >= escolhido["corte"]).astype(int)

    _extras = {}

    def cobertura(y_verdade, y_previsto):
        """A medida principal do contrato: das fontes que são o mesmo fato, quantas o 6B aponta. Guarda também a
        precisão, para o relatório (é a mesma e única abertura)."""
        _p, _c = precisao_e_cobertura(np.asarray(y_verdade, dtype=bool), np.asarray(y_previsto, dtype=bool))
        _extras.update(precisao=_p, cobertura=_c)
        return _c

    # A ABERTURA. O guard confere a impressão digital, conta a abertura e anota o resultado no PIPELINE.md.
    guard.final_test(prever, test, target="mesmo", metric_fn=cobertura,
                     state_dir=_cofre, pipeline_md=_cofre / "PIPELINE.md")

    # Na MESMA abertura: o contrato (6B contra a regra), a régua (só informação) e as fatias.
    _y = test["mesmo"].to_numpy(dtype=bool)
    _l = _linhas_com_nota.loc[test["id"]]
    _aponta_6b = prever(test[["id"]]).astype(bool)
    _aponta_regra = _l["regra"].isin(MESMO_DA_REGRA).to_numpy()
    _contrato = passa_no_contrato(_y, _aponta_6b, _aponta_regra, _l["grupo"])
    # A RÉGUA (plano B sem o modelo de linguagem): treinada no estudo e com o corte do simulado, como na etapa 9.
    from medidas import corte_com_precisao as _corte14
    _prep_r = montar_preprocessador(com_regra=False)
    _regua = _P14([("preproc", _prep_r), ("logistica", _LR14(C=0.1, class_weight="balanced", max_iter=3000))])
    _regua.fit(estudo[_prep_r.entradas], estudo["mesmo"])
    _prec_regra_sim, _ = precisao_e_cobertura(simulado["mesmo"], simulado["regra"].isin(MESMO_DA_REGRA))
    _corte_regua = _corte14(simulado["mesmo"], _regua.predict_proba(simulado[_prep_r.entradas])[:, 1],
                            _prec_regra_sim)["corte"]
    _aponta_regua = _regua.predict_proba(_l[_prep_r.entradas])[:, 1] >= _corte_regua

    _faixa = (pd.cut(_l["idade_dias"], [-10, 1, 30, 10**6], labels=["mesmo dia ou véspera", "2 a 30 dias",
                                                                     "mais de um mês"])
              .cat.add_categories("sem data").fillna("sem data").astype(str).to_numpy())
    _recortes = {
        "toda a prova": np.ones(len(_y), dtype=bool),
        "a regra comparou com cuidado": (_l["comparou"] == 1).to_numpy(),
        "só relevância": (_l["comparou"] == 0).to_numpy(),
        "sem os duvidosos (maioria de três)": (_l["origem"] != "maioria de três").to_numpy(),
        "sem os \"mesmo\" de fonte velha": ~_l["mesmo_de_fonte_velha"].to_numpy(dtype=bool),
        **{f"fonte: {v}": _faixa == v for v in ["mesmo dia ou véspera", "2 a 30 dias", "mais de um mês", "sem data"]},
    }
    _fatias = {}
    for _nome, _m in _recortes.items():
        _linha = {"fontes": int(_m.sum()), "do mesmo fato": int(_y[_m].sum())}
        for _quem, _ap in [("6B", _aponta_6b), ("regra", _aponta_regra), ("régua", _aponta_regua)]:
            _p, _c = precisao_e_cobertura(_y[_m], _ap[_m])
            _linha[f"{_quem}: acha de cada 10"] = round(10 * _c, 2)
            _linha[f"{_quem}: precisão"] = round(_p, 3)
        _fatias[_nome] = _linha
    _p6, _c6 = precisao_e_cobertura(_y, _aponta_6b)
    _pr, _cr = precisao_e_cobertura(_y, _aponta_regra)
    resultado_prova = {
        "fontes": int(len(_y)), "do mesmo fato": int(_y.sum()),
        "6B: acha de cada 10": round(10 * _c6, 2), "6B: precisão": round(_p6, 3),
        "regra: acha de cada 10": round(10 * _cr, 2), "regra: precisão": round(_pr, 3),
        "exigência 1: 6B menos regra, de cada 10": round(10 * _contrato["cobertura"]["diferenca"], 2),
        "exigência 1: faixa": [round(10 * _contrato["cobertura"]["de"], 2), round(10 * _contrato["cobertura"]["ate"], 2)],
        "exigência 1: passa": _contrato["acha_mais"],
        "exigência 2: precisão do 6B menos a da regra, em pontos": round(100 * _contrato["precisao"]["diferenca"], 1),
        "exigência 2: faixa, em pontos": [round(100 * _contrato["precisao"]["de"], 1),
                                          round(100 * _contrato["precisao"]["ate"], 1)],
        "exigência 2: passa": _contrato["nao_erra_demais"],
        "passa no contrato": _contrato["passa"],
        "por fatia": _fatias,
    }
    if not AMOSTRA_TESTE:
        _arquivo_resultado.parent.mkdir(exist_ok=True)
    _arquivo_resultado.write_text(json.dumps(resultado_prova, indent=1, ensure_ascii=False) + "\n",
                                  encoding="utf-8", newline="\n")
    aviso_prova = "Prova aberta agora, uma única vez. Resultado gravado em resultados/14_prova.json."
mo.vstack([mo.md(f"**{aviso_prova}**"), pd.Series({k: v for k, v in resultado_prova.items() if k != "por fatia"}),
           pd.DataFrame(resultado_prova["por fatia"]).T])

# Ciclo 2, etapa 15: preparar o 6B para o app (pedida pelo Matheus em 08/10/2026)

**Em português:** o app roda no celular, em Kotlin. Ele não abre o modelo do Python. Então tiramos do
6B tudo o que ele aprendeu (a média e a escala de cada número, um peso por informação, a nota de corte) e escrevemos
a **receita**: uma conta de somar e multiplicar que qualquer linguagem faz. Uma das informações do 6B, a nota do
modelo de linguagem, o celular não calcula (436 MB): ela vai precisar de um servidor, que você decidiu escolher
depois (seu PC ou uma das VPS). Enquanto não houver servidor, o app usa o **plano B**: a régua, a mesma conta sem a
nota do texto, com o seu próprio corte (na prova, 7,1 de cada 10 contra 7,6 do 6B). Os dois precisam da semelhança do
MPNet, que o app já pede ao Ollama no computador; sem ele, fica só a regra, como hoje.

O desenho do selo, decidido por você: **regra intacta** (onde a regra diz "mesmo acontecimento", nada muda) e **selo
próprio só para fonte do mesmo dia ou da véspera**; nas outras fontes que o modelo aponta, "mesmo assunto, confira a
data".

| O que fazemos | Em português | Em termos técnicos |
|---|---|---|
| **As duas receitas** | as contas do 6B e do plano B, cada uma num arquivo pequeno | coeficientes, médias, escalas e corte em JSON |
| **A prova das receitas** | a nota feita à mão tem de ser igual à do modelo em todas as fontes do simulado | conferência numérica, diferença máxima |
| **Os arquivos completos** | os modelos inteiros salvos e abertos de novo dão as mesmas notas | artefato joblib recarregado |
| **Os casos de conferência** | fontes do simulado com a nota e o selo certos ao lado, para o teste do app | casos dourados |
| **O selo no simulado** | quanto acerta cada selo que o leitor veria | precisão por selo, por notícia |

Tudo no **simulado**; a prova já foi gasta e não é aberta de novo. A regra da data foi escolhida olhando a prova (e a
etapa 13), então o número do selo na prova não seria uma medida independente: fica de fora de propósito.

**O que eu espero, escrito antes de rodar:** as duas receitas e os dois arquivos recarregados dão a mesma nota dos
modelos (diferença menor que 0,000000001). No simulado, "provável mesmo fato" acerta de 8 a 9,5 de cada 10 (o 6B
acerta 94% no mesmo dia, mas as fontes que a regra já pega saem da conta, e sobram as difíceis); "mesmo assunto,
confira a data" acerta de 3 a 6 de cada 10 (é o aviso, não a confirmação).

In [ ]:
import json as _json15
import joblib
from sklearn.linear_model import LogisticRegression as _LR15
from sklearn.pipeline import Pipeline as _P15
from medidas import corte_com_precisao as _corte15
from modelo_para_o_app import exportar, nota_a_mao

# Onde gravar: ml_pipeline/app/ (no teste pequeno, numa pasta temporária que some).
pasta_app = PASTA_SAIDA / "app"
pasta_app.mkdir(exist_ok=True)

# O PLANO B: a régua da etapa 9 (logística C=0,1 sobre as 34 informações, sem a nota do texto), treinada no estudo,
# com o corte do contrato escolhido no simulado (erra o mesmo que a regra ali).
_prep_b = montar_preprocessador(com_regra=False)
plano_b = _P15([("preproc", _prep_b), ("logistica", _LR15(C=0.1, class_weight="balanced", max_iter=3000))])
plano_b.fit(estudo[_prep_b.entradas], estudo["mesmo"])
entradas_plano_b = list(_prep_b.entradas)
nota_plano_b = plano_b.predict_proba(simulado[entradas_plano_b])[:, 1]
_prec_regra, _ = precisao_e_cobertura(simulado["mesmo"], simulado["regra"].isin(MESMO_DA_REGRA))
corte_plano_b = _corte15(simulado["mesmo"], nota_plano_b, _prec_regra)["corte"]

# As entradas de cada um, no simulado: o 6B recebe também a nota do texto.
_sim_6b = simulado.assign(nota_texto=nota_texto_simulado)[escolhido["entradas"]]
_sim_b = simulado[entradas_plano_b]

conferencia_15 = {}
receitas = {}
for _nome, _modelo, _tabela, _nota, _corte, _arq in [
    ("6B", modelo_escolhido, _sim_6b, nota, escolhido["corte"], "decisor"),
    ("plano B", plano_b, _sim_b, nota_plano_b, corte_plano_b, "decisor_plano_b"),
]:
    # 1. A RECEITA: tudo o que o modelo aprendeu, em números simples.
    receitas[_nome] = exportar(_modelo, _corte)
    # 2. O ARQUIVO COMPLETO (pré-processamento + modelo juntos), salvo e aberto de novo (fora do git: *.joblib).
    joblib.dump(_modelo, pasta_app / f"{_arq}.joblib")
    _recarregada = joblib.load(pasta_app / f"{_arq}.joblib").predict_proba(_tabela)[:, 1]
    # 3. A NOTA À MÃO, fonte por fonte, com a receita: é a conta que o Kotlin vai copiar.
    _mao = np.array([nota_a_mao(receitas[_nome], _l) for _l in _tabela.to_dict("records")])
    # 4. A CONFERÊNCIA: as três notas têm de ser iguais.
    conferencia_15[_nome] = {
        "fontes conferidas": int(len(_nota)),
        "maior diferença: receita x modelo": float(np.max(np.abs(_mao - _nota))),
        "maior diferença: arquivo recarregado x modelo": float(np.max(np.abs(_recarregada - _nota))),
        "decisões diferentes no corte": int(np.sum((_mao >= _corte) != (_nota >= _corte))),
        "corte": float(_corte),
    }
    assert conferencia_15[_nome]["maior diferença: receita x modelo"] < 1e-9, f"a receita do {_nome} NÃO reproduz"
    assert conferencia_15[_nome]["maior diferença: arquivo recarregado x modelo"] < 1e-12, "o recarregado NÃO reproduz"
    # 5. Grava a receita (versionada no git; no teste pequeno, vai para a pasta temporária).
    receitas[_nome]["o_que_e"] += (" | 6B: precisa da nota_texto (BERTimbau, servidor)" if _nome == "6B"
                                   else " | plano B: sem a nota do texto (precisa só do MPNet, pelo Ollama)")
    (pasta_app / f"{_arq}.json").write_text(_json15.dumps(receitas[_nome], indent=1, ensure_ascii=False) + "\n",
                                            encoding="utf-8", newline="\n")
conferencia_15

In [ ]:
import json as _json15b
from modelo_para_o_app import selo

# OS CASOS DE CONFERÊNCIA: fontes do simulado que cobrem cada valor de cada categoria, as sem data, e cada selo do
# 6B e do plano B. A escolha é fixa (o menor id de cada caso), para dar o mesmo arquivo em qualquer máquina.
_tab = simulado.assign(nota_6b=nota, nota_texto=nota_texto_simulado, nota_b=nota_plano_b)
_tab["selo_6b"] = [selo(r, n, escolhido["corte"], d) for r, n, d in zip(_tab["regra"], _tab["nota_6b"], _tab["idade_dias"])]
_tab["selo_b"] = [selo(r, n, corte_plano_b, d) for r, n, d in zip(_tab["regra"], _tab["nota_b"], _tab["idade_dias"])]
_ids = set()
for _coluna in receitas["6B"]["categorias"]:                             # um caso por valor de cada categoria
    for _valor, _grupo in _tab.groupby(_coluna):
        _ids.add(min(_grupo.index))
_ids |= set(sorted(_tab.index[_tab["idade_dias"].isna()])[:3])          # três sem data
for _coluna in ["selo_6b", "selo_b"]:                                    # três de cada selo, dos dois
    for _s, _grupo in _tab.groupby(_coluna):
        _ids |= set(sorted(_grupo.index)[:3])

def _simples(v):
    return None if pd.isna(v) else (v.item() if hasattr(v, "item") else v)

_colunas = list(dict.fromkeys(list(escolhido["entradas"]) + entradas_plano_b))
casos_15 = []
for _id in sorted(_ids):
    _l = _tab.loc[_id]
    casos_15.append({
        "id": str(_id),
        "afirmacao": _l["afirmacao"], "titulo": _l["titulo"],            # para o app conferir cosseno e sobreposição
        "entradas": {_c: _simples(_l[_c]) for _c in _colunas},
        "regra": _l["regra"],
        "nota_esperada_6b": round(float(_l["nota_6b"]), 9), "selo_esperado_6b": _l["selo_6b"],
        "nota_esperada_plano_b": round(float(_l["nota_b"]), 9), "selo_esperado_plano_b": _l["selo_b"],
    })
(pasta_app / "casos_de_conferencia.json").write_text(
    _json15b.dumps(casos_15, indent=1, ensure_ascii=False) + "\n", encoding="utf-8", newline="\n")

# O SELO NO SIMULADO: o que o leitor veria, por notícia, e quanto cada selo acerta.
_y = _tab["mesmo"].to_numpy(dtype=bool)
_noticias = _tab["grupo"].nunique()
_linhas = []
for _quem, _col in [("6B", "selo_6b"), ("plano B", "selo_b")]:
    _selos = _tab[_col]
    for _nome, _mascara in {
        "a regra diz mesmo (nada muda)": _selos.str.startswith("regra: ") & _tab["regra"].isin(receitas["6B"]["mesmo_da_regra"]),
        "provável mesmo fato": _selos == "provável mesmo fato",
        "mesmo assunto, confira a data": _selos == "mesmo assunto, confira a data",
    }.items():
        _m = _mascara.to_numpy()
        _linhas.append({"quem": _quem, "selo": _nome,
                        "mostradas por notícia": round(_m.sum() / _noticias, 2),
                        "certas por notícia": round((_m & _y).sum() / _noticias, 2),
                        "acerta de cada 10": round(10 * (_m & _y).sum() / max(_m.sum(), 1), 1)})
selo_no_simulado = pd.DataFrame(_linhas).set_index(["quem", "selo"])
selo_no_simulado

In [ ]:
# OS RESULTADOS EM ARQUIVO, para a sessão de IA ler do disco.
_pasta = PASTA_SAIDA / "resultados"
_pasta.mkdir(exist_ok=True)
(_pasta / "15_app.json").write_text(json.dumps({
    "etapa": 15, "conferencia": conferencia_15, "casos_de_conferencia": len(casos_15),
    "selo_no_simulado": {" | ".join(k): v for k, v in selo_no_simulado.to_dict(orient="index").items()},
}, indent=1, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, "item") else str(o)) + "\n",
    encoding="utf-8", newline="\n")
mo.md(f"Receitas, arquivos e {len(casos_15)} casos de conferência gravados em `app/`; resumo em `resultados/15_app.json`.")

# Ciclo 2, etapa 16: vigiar e refazer

**Em português:** a etapa 16 é o plano para depois que o 6B estiver no app: o que vigiar, quando ajustar a nota de
corte e quando refazer tudo do zero. O plano inteiro, com o que precisa ser verdade para cada passo, está no
`PIPELINE.md` ("Etapa 16" no diário, "Ciclo 3: o plano" e "Relatório do ciclo 2"); o painel, na primeira célula,
mostra a história.

| O que | Em português | Em termos técnicos |
|---|---|---|
| **Vigiar sem rótulo** | quantos selos por notícia (2,1 "provável mesmo fato" e 1,1 "confira a data" no simulado), se o 6B deixa de achar o que a regra acha, se as entradas mudam de cara, quantas vezes o celular caiu para o plano B | deriva das previsões e das entradas |
| **Vigiar com rótulo** | rotular umas 100 manchetes de cada época nova e medir o mesmo contrato contra a regra | amostra nova, mesma régua |
| **Ajustar o corte** | se o 6B só errar mais, reescolher a nota de corte numa metade da amostra nova e medir na outra | recalibração do limiar |
| **Refazer** | se nem com o corte novo ele ganhar da regra, o pipeline recomeça da etapa 1 | retreino pelo mesmo pipeline |
| **O ciclo 3** | revisar às cegas os rótulos suspeitos, entender por que a prova foi mais difícil, um modelo de linguagem que caiba no celular, calibrar a nota, mais fontes de 2 a 30 dias | plano por ordem do maior buraco |